# Deliverable 4 — Apache Airflow Orchestration

This notebook implements the orchestration layer of the Restaurant Delivery Lakehouse using
Apache Airflow.

The workflow coordinates the execution of the pipeline by loading the Silver table, validating
data quality with a real Great Expectations gate, generating the Gold layer, and recording
OpenLineage events per stage — and it is actually **run twice** (clean data, then bad data) so
the halt-on-failure behaviour the rubric asks for is captured as real output, not just described.


## Architecture
```text
Silver Delta Table
        ↓
Quality Gate  (Great Expectations)
        ↓
Gold Layer   (real aggregate)
        ↓
OpenLineage  (START / COMPLETE / FAIL per stage)
```

Task dependency: `load_silver >> quality_gate >> build_gold >> emit_lineage`

`quality_gate`'s default trigger rule (`all_success`) means that if it fails, Airflow marks
`build_gold` and `emit_lineage` as `upstream_failed` and their code never runs — that is the
mechanism that proves a failed quality gate halts the pipeline.


## 1. Environment Setup

Install Airflow pinned to the constraints file for this Colab's Python version, plus
PySpark/Delta (for the Silver/Gold Delta tables), Great Expectations (the quality gate), and
`openlineage-python` (the lineage events).


In [1]:
import sys
PY_VERSION = f"{sys.version_info.major}.{sys.version_info.minor}"
AIRFLOW_VERSION = "2.9.3"
CONSTRAINT_URL = (
    f"https://raw.githubusercontent.com/apache/airflow/constraints-{AIRFLOW_VERSION}/"
    f"constraints-{PY_VERSION}.txt"
)
print("Python:", PY_VERSION)
print("Constraint file:", CONSTRAINT_URL)

!pip install -q "apache-airflow=={AIRFLOW_VERSION}" --constraint "{CONSTRAINT_URL}"
!pip install -q pyspark==3.5.0 delta-spark==3.2.0 great_expectations openlineage-python


Python: 3.12
Constraint file: https://raw.githubusercontent.com/apache/airflow/constraints-2.9.3/constraints-3.12.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.0/233.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.8/49.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [2]:
import os

# A dedicated, disposable Airflow home inside the Colab VM.
os.environ["AIRFLOW_HOME"] = "/content/airflow"
os.environ["AIRFLOW__CORE__DAGS_FOLDER"] = "/content/airflow/dags"
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
os.environ["AIRFLOW__CORE__EXECUTOR"] = "SequentialExecutor"
os.environ["AIRFLOW__LOGGING__LOGGING_LEVEL"] = "INFO"

os.makedirs("/content/airflow/dags", exist_ok=True)

!airflow db migrate


# Remove stale lineage evidence from previous notebook executions.
for path in [
    "/content/openlineage_events.jsonl",
]:
    if os.path.exists(path):
        os.remove(path)


DB: sqlite:////content/airflow/airflow.db
Performing upgrade to the metadata database sqlite:////content/airflow/airflow.db
[2026-07-23T08:15:11.136+0000] {migration.py:215} INFO - Context impl SQLiteImpl.
[2026-07-23T08:15:11.139+0000] {migration.py:218} INFO - Will assume non-transactional DDL.
[2026-07-23T08:15:11.143+0000] {migration.py:215} INFO - Context impl SQLiteImpl.
[2026-07-23T08:15:11.143+0000] {migration.py:218} INFO - Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running stamp_revision  -> 686269002441
Database migrating done!


## 2. Pipeline Tasks

Each task below does real work — a real Delta write/read, a real Great Expectations
validation, a real Delta aggregate, and real `openlineage-python` events — and is written to
the Airflow DAGs folder so `airflow dags test` can actually import and execute it (a DAG
object built only in the notebook's own memory is not discoverable by the Airflow CLI).

The Great Expectations checkpoint and the OpenLineage emitter below follow the exact same
`Checkpoint` / `ValidationDefinition` and `OpenLineageClient` / `FileTransport` calls verified
working in **Day 4 Lab — Data Quality, Governance, and Lineage**, just pointed at the Silver
Delta table instead of the Kaggle retail CSV.

Bad data can be injected through the DAG run's `conf` (`{"inject_bad_data": true}`), which
`load_silver` uses to add one deliberately invalid row to Silver — that is what `quality_gate`
is supposed to catch.


In [3]:
%%writefile /content/airflow/dags/restaurant_delivery_pipeline.py
"""
Restaurant Delivery Lakehouse — orchestration DAG.

Tasks: load_silver -> quality_gate -> build_gold -> emit_lineage

quality_gate raises on failure, so Airflow's default trigger_rule ("all_success") makes
build_gold and emit_lineage upstream_failed automatically whenever the gate fails.
"""
import json
import os
import shutil
import uuid
from datetime import datetime, timezone

from airflow import DAG
from airflow.operators.python import PythonOperator

BRONZE_PATH = "/content/delivery_data/bronze"
SILVER_PATH = "/content/delivery_data/silver"
GOLD_PATH = "/content/delivery_data/gold"
LINEAGE_LOG_PATH = "/content/openlineage_events.jsonl"


# ---------------------------------------------------------------------------
# OpenLineage helper — same OpenLineageClient / FileTransport / RunEvent calls
# verified working in Day 4 Lab's emit_real_openlineage_events(), reused here
# per-stage instead of once per pipeline run.
# ---------------------------------------------------------------------------
def _emit(run_id: str, job_name: str, state: str, detail: str = None):
    from openlineage.client import OpenLineageClient
    from openlineage.client.transport.file import FileConfig, FileTransport
    from openlineage.client.event_v2 import RunEvent, RunState, Run, Job

    os.makedirs(os.path.dirname(LINEAGE_LOG_PATH), exist_ok=True)
    transport = FileTransport(FileConfig(log_file_path=LINEAGE_LOG_PATH))
    client = OpenLineageClient(transport=transport)

    job = Job(namespace="restaurant_delivery_pipeline", name=job_name)
    # OpenLineage requires runId to be a real UUID. Airflow's own run_id
    # ("manual__2026-01-01T00:00:00+00:00") is not one, so derive a stable
    # UUID5 from it -- deterministic per DAG run, and still groupable in
    # the verification step below.
    lineage_run_id = str(uuid.uuid5(uuid.NAMESPACE_URL, run_id))
    run = Run(runId=lineage_run_id)
    now = datetime.now(timezone.utc).isoformat()

    client.emit(RunEvent(
        eventType=getattr(RunState, state),
        eventTime=now,
        run=run,
        job=job,
        producer="https://github.com/sdaia/modern-data-engineering-lab",
    ))
    suffix = f" -> {detail}" if detail else ""
    print(f"[OpenLineage] {state:8s} {job_name}{suffix}")


# ---------------------------------------------------------------------------
# Task 1 — load_silver: build Bronze, clean into Silver
# ---------------------------------------------------------------------------
def load_silver(**context):
    from delta import configure_spark_with_delta_pip
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        DoubleType, IntegerType, StructField, StructType, StringType,
    )

    run_id = context["run_id"]
    inject_bad_data = (context["dag_run"].conf or {}).get("inject_bad_data", False)
    _emit(run_id, "load_silver", "START")

    if os.path.exists("/content/delivery_data"):
        shutil.rmtree("/content/delivery_data")

    builder = (
        SparkSession.builder
        .appName("RestaurantDeliveryLakehouse")
        .master("local[*]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    schema = StructType([
        StructField("delivery_id", StringType(), nullable=False),
        StructField("customer_id", StringType(), nullable=True),
        StructField("restaurant_name", StringType(), nullable=True),
        StructField("order_amount", DoubleType(), nullable=True),
        StructField("delivery_minutes", IntegerType(), nullable=True),
        StructField("city_zone", StringType(), nullable=True),
        StructField("delivery_status", StringType(), nullable=True),
    ])

    batch = [
        ("DEL101", "CUS101", "Najd Kitchen", 86.50, 28, " North ", "COMPLETED"),
        ("DEL102", "CUS102", "Burger House", 42.00, 18, "East", "Completed"),
        ("DEL103", "CUS103", "Sushi Corner", 110.25, 35, "West", "PREPARING"),
        ("DEL104", "CUS104", "Pasta Villa", 63.75, 25, "South", "Completed"),
    ]
    bronze_df = spark.createDataFrame(batch, schema)
    bronze_df.write.format("delta").mode("overwrite").save(BRONZE_PATH)

    silver_df = (
        spark.read.format("delta").load(BRONZE_PATH)
        .withColumn("restaurant_name", F.trim(F.col("restaurant_name")))
        .withColumn("city_zone", F.upper(F.trim(F.col("city_zone"))))
        .withColumn("delivery_status", F.lower(F.trim(F.col("delivery_status"))))
    )

    if inject_bad_data:
        # Deliberately invalid row: negative order_amount, blank city_zone.
        bad_row = spark.createDataFrame(
            [("DEL999", "CUS999", "Broken Kitchen", -40.0, 20, "", "completed")],
            schema,
        ).withColumn("restaurant_name", F.trim(F.col("restaurant_name")))
        silver_df = silver_df.unionByName(bad_row)

    silver_df.write.format("delta").mode("overwrite").save(SILVER_PATH)

    print("Silver table loaded:")
    spark.read.format("delta").load(SILVER_PATH).show(truncate=False)
    spark.stop()

    _emit(run_id, "load_silver", "COMPLETE")


# ---------------------------------------------------------------------------
# Task 2 — quality_gate: real Great Expectations checkpoint
#
# Same Checkpoint / ValidationDefinition / checkpoint.run() calls verified
# working in Day 4 Lab's run_great_expectations_checkpoint(), pointed at the
# Silver Delta table's columns instead of the retail CSV's columns.
# ---------------------------------------------------------------------------
def quality_gate(**context):
    import glob
    import pandas as pd
    import great_expectations as gx
    import great_expectations.expectations as gxe

    run_id = context["run_id"]
    _emit(run_id, "quality_gate", "START")

    parquet_files = glob.glob(f"{SILVER_PATH}/*.parquet")
    silver_pdf = pd.concat([pd.read_parquet(p) for p in parquet_files], ignore_index=True)

    gx_context = gx.get_context(mode="ephemeral")
    data_source = gx_context.data_sources.add_pandas("silver_layer")
    data_asset = data_source.add_dataframe_asset(name="restaurant_delivery_silver")
    batch_definition = data_asset.add_batch_definition_whole_dataframe("silver_batch")

    suite = gx_context.suites.add(gx.ExpectationSuite(name="silver_layer_suite"))
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="delivery_id"))
    suite.add_expectation(gxe.ExpectColumnValueLengthsToBeBetween(column="city_zone", min_value=1))
    suite.add_expectation(
        gxe.ExpectColumnValuesToBeBetween(column="order_amount", min_value=0.0001)
    )

    validation_definition = gx_context.validation_definitions.add(
        gx.ValidationDefinition(
            name="silver_layer_validation",
            data=batch_definition,
            suite=suite,
        )
    )
    checkpoint = gx_context.checkpoints.add(
        gx.Checkpoint(
            name="silver_layer_checkpoint",
            validation_definitions=[validation_definition],
        )
    )
    result = checkpoint.run(batch_parameters={"dataframe": silver_pdf})

    print(f"[GX] Real Great Expectations checkpoint success={result.success}")
    failed_expectations = []
    for run_result in result.run_results.values():
        for r in run_result["results"]:
            status = "PASSED" if r["success"] else "FAILED"
            print(f"  [GX] {status} {r['expectation_config']['type']}")
            if not r["success"]:
                failed_expectations.append(r["expectation_config"]["type"])

    if not result.success:
        detail = f"{len(failed_expectations)} expectation(s) failed: {failed_expectations}"
        _emit(run_id, "quality_gate", "FAIL", detail=detail)
        raise ValueError(f"Quality Gate FAILED: {detail}")

    print("Quality Gate PASSED.")
    _emit(run_id, "quality_gate", "COMPLETE")


# ---------------------------------------------------------------------------
# Task 3 — build_gold: real aggregate (only runs if quality_gate passed)
# ---------------------------------------------------------------------------
def build_gold(**context):
    from delta import configure_spark_with_delta_pip
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    run_id = context["run_id"]
    _emit(run_id, "build_gold", "START")

    builder = (
        SparkSession.builder
        .appName("RestaurantDeliveryLakehouse")
        .master("local[*]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    gold_df = (
        spark.read.format("delta").load(SILVER_PATH)
        .groupBy("city_zone")
        .agg(
            F.count("delivery_id").alias("total_deliveries"),
            F.round(F.sum("order_amount"), 2).alias("total_order_value"),
            F.round(F.avg("delivery_minutes"), 2).alias("average_delivery_minutes"),
        )
        .orderBy("city_zone")
    )
    gold_df.write.format("delta").mode("overwrite").save(GOLD_PATH)

    print("Gold layer created:")
    spark.read.format("delta").load(GOLD_PATH).show(truncate=False)
    spark.stop()

    _emit(run_id, "build_gold", "COMPLETE")


# ---------------------------------------------------------------------------
# Task 4 — emit_lineage: pipeline-level summary event (only if build_gold ran)
# ---------------------------------------------------------------------------
def emit_lineage(**context):
    run_id = context["run_id"]
    _emit(run_id, "emit_lineage", "START")
    print("Emitting OpenLineage COMPLETE event for the full pipeline run...")
    _emit(run_id, "restaurant_delivery_pipeline", "COMPLETE")
    _emit(run_id, "emit_lineage", "COMPLETE")


with DAG(
    dag_id="restaurant_delivery_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    description="Restaurant Delivery Lakehouse Pipeline",
) as dag:

    load_silver_task = PythonOperator(
        task_id="load_silver",
        python_callable=load_silver,
    )

    quality_gate_task = PythonOperator(
        task_id="quality_gate",
        python_callable=quality_gate,
    )

    build_gold_task = PythonOperator(
        task_id="build_gold",
        python_callable=build_gold,
    )

    lineage_task = PythonOperator(
        task_id="emit_lineage",
        python_callable=emit_lineage,
    )

    load_silver_task >> quality_gate_task >> build_gold_task >> lineage_task


Writing /content/airflow/dags/restaurant_delivery_pipeline.py


In [4]:
!airflow dags list-import-errors
!airflow dags list | grep restaurant_delivery_pipeline


No data found
restaurant_delivery_pipeline | /content/airflow/dags/restaurant_delivery_pipeline.py | airflow | None     


## 3. Run 1 — Happy Path

Clean data. All four tasks should succeed, including a real `emit_lineage` COMPLETE event for
the whole pipeline.


In [5]:
!airflow dags test restaurant_delivery_pipeline 2026-01-01


[2026-07-23T08:15:31.787+0000] {dagbag.py:545} INFO - Filling up the DagBag from /content/airflow/dags
[2026-07-23T08:15:31.882+0000] {dag.py:4199} INFO - dagrun id: restaurant_delivery_pipeline
[2026-07-23T08:15:31.899+0000] {dag.py:4215} INFO - created dagrun <DagRun restaurant_delivery_pipeline @ 2026-01-01 00:00:00+00:00: manual__2026-01-01T00:00:00+00:00, state:running, queued_at: None. externally triggered: False>
[2026-07-23T08:15:31.972+0000] {dag.py:4161} INFO - [DAG TEST] starting task_id=load_silver map_index=-1
[2026-07-23T08:15:31.973+0000] {dag.py:4164} INFO - [DAG TEST] running task <TaskInstance: restaurant_delivery_pipeline.load_silver manual__2026-01-01T00:00:00+00:00 [scheduled]>
[2026-07-23 08:15:32,128] {taskinstance.py:2648} INFO - Exporting env vars: AIRFLOW_CTX_DAG_OWNER='airflow' AIRFLOW_CTX_DAG_ID='restaurant_delivery_pipeline' AIRFLOW_CTX_TASK_ID='load_silver' AIRFLOW_CTX_EXECUTION_DATE='2026-01-01T00:00:00+00:00' AIRFLOW_CTX_TRY_NUMBER='1' AIRFLOW_CTX_DAG_RU

In [6]:
!airflow tasks states-for-dag-run restaurant_delivery_pipeline 2026-01-01


              | execution_d |              |         |            |             
dag_id        | ate         | task_id      | state   | start_date | end_date    
==============+=============+==============+=========+============+=============
restaurant_de | 2026-01-01T | load_silver  | success |            | 2026-07-23T0
livery_pipeli | 00:00:00+00 |              |         |            | 8:16:26.1949
ne            | :00         |              |         |            | 42+00:00    
restaurant_de | 2026-01-01T | quality_gate | success |            | 2026-07-23T0
livery_pipeli | 00:00:00+00 |              |         |            | 8:16:32.9460
ne            | :00         |              |         |            | 54+00:00    
restaurant_de | 2026-01-01T | build_gold   | success |            | 2026-07-23T0
livery_pipeli | 00:00:00+00 |              |         |            | 8:16:51.3811
ne            | :00         |              |         |            | 89+00:00    
restaurant_de | 2026-01-01T 

## 4. Run 2 — Bad Data (Halt-on-Failure Path)

`inject_bad_data=true` makes `load_silver` add one row with a negative `order_amount` and a
blank `city_zone`. `quality_gate` must catch it, fail, and stop `build_gold` and `emit_lineage`
from ever running.


In [7]:
!airflow dags test restaurant_delivery_pipeline 2026-01-02 --conf '{"inject_bad_data": true}'


[2026-07-23T08:16:58.552+0000] {dagbag.py:545} INFO - Filling up the DagBag from /content/airflow/dags
[2026-07-23T08:16:58.602+0000] {dag.py:4199} INFO - dagrun id: restaurant_delivery_pipeline
[2026-07-23T08:16:58.613+0000] {dag.py:4215} INFO - created dagrun <DagRun restaurant_delivery_pipeline @ 2026-01-02 00:00:00+00:00: manual__2026-01-02T00:00:00+00:00, state:running, queued_at: None. externally triggered: False>
[2026-07-23T08:16:58.647+0000] {dag.py:4161} INFO - [DAG TEST] starting task_id=load_silver map_index=-1
[2026-07-23T08:16:58.647+0000] {dag.py:4164} INFO - [DAG TEST] running task <TaskInstance: restaurant_delivery_pipeline.load_silver manual__2026-01-02T00:00:00+00:00 [scheduled]>
[2026-07-23 08:16:58,728] {taskinstance.py:2648} INFO - Exporting env vars: AIRFLOW_CTX_DAG_OWNER='airflow' AIRFLOW_CTX_DAG_ID='restaurant_delivery_pipeline' AIRFLOW_CTX_TASK_ID='load_silver' AIRFLOW_CTX_EXECUTION_DATE='2026-01-02T00:00:00+00:00' AIRFLOW_CTX_TRY_NUMBER='1' AIRFLOW_CTX_DAG_RU

In [8]:
!airflow tasks states-for-dag-run restaurant_delivery_pipeline 2026-01-02


            | execution_d |            |             |            |             
dag_id      | ate         | task_id    | state       | start_date | end_date    
============+=============+============+=============+============+=============
restaurant_ | 2026-01-02T | load_silve | success     |            | 2026-07-23T0
delivery_pi | 00:00:00+00 | r          |             |            | 8:17:54.1982
peline      | :00         |            |             |            | 08+00:00    
restaurant_ | 2026-01-02T | quality_ga | failed      |            | 2026-07-23T0
delivery_pi | 00:00:00+00 | te         |             |            | 8:17:58.7604
peline      | :00         |            |             |            | 87+00:00    
restaurant_ | 2026-01-02T | build_gold | upstream_fa | 2026-07-23 | 2026-07-23T0
delivery_pi | 00:00:00+00 |            | iled        | T08:17:58. | 8:17:58.7871
peline      | :00         |            |             | 787124+00: | 24+00:00    
            |             | 

## 5. Verifying the Lineage Trail

Reading the emitted `openlineage_events.jsonl` back, grouped by Airflow's own `run_id` for each
of the two runs above, shows exactly which stages emitted events — and, for the bad-data run,
that `build_gold` and `emit_lineage` emitted **none**, because they never ran.


In [11]:
import glob
import json

# OpenLineage created timestamped JSON files instead of one base JSONL file.
lineage_files = sorted(
    glob.glob("/content/openlineage_events.jsonl-*.json")
)

print("Lineage files found:", len(lineage_files))

if not lineage_files:
    raise FileNotFoundError(
        "No OpenLineage event files were found. "
        "Run the happy-path and bad-data DAG tests first."
    )

events = []

for file_path in lineage_files:
    with open(file_path, "r", encoding="utf-8") as lineage_file:
        content = lineage_file.read().strip()

    if not content:
        continue

    # Each timestamped file normally contains one JSON event.
    # This fallback also handles concatenated JSON objects.
    decoder = json.JSONDecoder()
    position = 0

    while position < len(content):
        next_object = content.find("{", position)

        if next_object == -1:
            break

        try:
            event, next_position = decoder.raw_decode(
                content,
                next_object
            )

            if isinstance(event, dict):
                events.append(event)

            position = next_position

        except json.JSONDecodeError:
            position = next_object + 1

print("\nTotal events captured:", len(events))

for event in events:
    print(
        event.get("eventType", "UNKNOWN"),
        "|",
        event.get("job", {}).get("name", "UNKNOWN"),
        "|",
        event.get("eventTime", "UNKNOWN")
    )

event_types = {
    event.get("eventType")
    for event in events
    if event.get("eventType")
}

print("\nEvent types found:", event_types)

assert "START" in event_types, "No START event was found."
assert "COMPLETE" in event_types, "No COMPLETE event was found."
assert "FAIL" in event_types, "No FAIL event was found."

print(
    "\n✅ START, COMPLETE, and FAIL "
    "lineage events were captured."
)

Lineage files found: 13

Total events captured: 13
START | load_silver | 2026-07-23T08:15:33.008204+00:00
COMPLETE | load_silver | 2026-07-23T08:16:26.185542+00:00
START | quality_gate | 2026-07-23T08:16:32.663405+00:00
COMPLETE | quality_gate | 2026-07-23T08:16:32.941320+00:00
START | build_gold | 2026-07-23T08:16:33.036180+00:00
COMPLETE | build_gold | 2026-07-23T08:16:51.374763+00:00
START | emit_lineage | 2026-07-23T08:16:51.461114+00:00
COMPLETE | restaurant_delivery_pipeline | 2026-07-23T08:16:51.462531+00:00
COMPLETE | emit_lineage | 2026-07-23T08:16:51.463770+00:00
START | load_silver | 2026-07-23T08:16:59.339653+00:00
COMPLETE | load_silver | 2026-07-23T08:17:54.189850+00:00
START | quality_gate | 2026-07-23T08:17:58.638344+00:00
FAIL | quality_gate | 2026-07-23T08:17:58.755157+00:00

Event types found: {'FAIL', 'START', 'COMPLETE'}

✅ START, COMPLETE, and FAIL lineage events were captured.


In [12]:
print("DAG Name:")
print("restaurant_delivery_pipeline")
print()
print("Tasks:")
for task_id in ["load_silver", "quality_gate", "build_gold", "emit_lineage"]:
    print("-", task_id)


DAG Name:
restaurant_delivery_pipeline

Tasks:
- load_silver
- quality_gate
- build_gold
- emit_lineage


# Technical Documentation

## DAG Dependency Graph

```text
load_silver → quality_gate → build_gold → emit_lineage
```

The default Airflow `all_success` trigger rule is intentionally used. When `quality_gate` raises an exception, `build_gold` and `emit_lineage` are marked `upstream_failed` and cannot execute.

## Real Components

- Apache Airflow DAG and `PythonOperator` tasks
- PySpark and Delta Lake Bronze/Silver/Gold work
- Great Expectations checkpoint inside the quality-gate task
- OpenLineage events emitted from pipeline stages

## Failure Demonstration

The second DAG test injects a negative order amount and blank city zone. The Great Expectations checkpoint fails, the quality-gate task raises an exception, and the downstream tasks are blocked.

## Runtime Files

The DAG source is written to:

```text
/content/airflow/dags/restaurant_delivery_pipeline.py
```

Lineage evidence is written to:

```text
/content/openlineage_events.jsonl
```